# Clase 2. El dilema de los bandidos: exploración y explotación

**DIH DATAlife — Curso de Aprendizaje por Refuerzo desde cero**

> Guion de clase: usa este notebook en directo. Cada celda de código es pequeña y se ejecuta paso a paso, para que puedas parar, comentar el resultado y responder preguntas antes de seguir.

## Qué vas a ver hoy

- El dilema entre explorar y explotar, con ejemplos cotidianos
- El problema del bandido multi-brazo
- Cómo estimar el valor de una acción
- La estrategia epsilon-greedy (ε-greedy)
- La estrategia Upper Confidence Bound (UCB)
- Una pincelada de Thompson Sampling
- Código en vivo: epsilon-greedy contra UCB, con gráfica comparativa


## Uno. El dilema entre exploración y explotación

> Guion: abre la clase con la pregunta del restaurante. Deja que el grupo responda antes de continuar.

Imagina que te mudas a una ciudad nueva. Tienes un restaurante que te gusta a cinco minutos de casa: sabes qué vas a comer y sabes que te va a gustar. Pero, a dos calles, hay un local que nunca has probado. ¿Vuelves a tu opción segura o te arriesgas a descubrir algo mejor?

Esa pregunta tan cotidiana es el mismo problema que enfrenta un equipo médico al elegir entre un tratamiento consolidado y uno experimental, o una plataforma digital al decidir qué anuncio mostrarte. En los tres casos hay una tensión entre dos impulsos:

- **Explotar** lo que ya sabes que funciona
- **Explorar** para averiguar si hay algo mejor

En Aprendizaje por Refuerzo (RL, por sus siglas en inglés), a esta tensión se le llama **dilema entre exploración y explotación**, y es uno de los aspectos más importantes de todo el campo.

Recuerda las piezas básicas de RL de la clase 1: un agente que actúa, un entorno que responde, una política que decide qué hacer y una recompensa que mide si esa decisión fue buena. Hoy vas a ver cómo un agente decide, paso a paso, entre repetir lo conocido y probar algo nuevo, usando la versión más simple posible de un problema de decisión: el **bandido multi-brazo**.


## Dos. El problema del bandido multi-brazo

> Guion: dibuja en la pizarra una fila de máquinas tragaperras si tienes tiempo. Es una imagen que se queda.

En un casino, una máquina tragaperras se conoce, en inglés, como "one-armed bandit" (bandido de un brazo), porque tiene una sola palanca y "te roba" el dinero poco a poco. Ahora imagina una fila de varias de estas máquinas, una junto a otra. Cada máquina paga premios distintos: unas pagan mucho, con frecuencia; otras casi nunca pagan nada. El problema es que tú no sabes cuál es cuál. Tu única forma de averiguarlo es tirar de las palancas y observar qué ocurre. Ese escenario completo se llama **bandido multi-brazo** (multi-armed bandit), y da nombre a esta familia de problemas.

Puntos clave:

- Cada palanca representa una acción posible
- Cada acción tiene un **valor verdadero**: la recompensa media que obtendrías si la eligieras infinitas veces
- Ese valor existe, pero tú no lo conoces; solo puedes estimarlo tirando de la palanca
- Cada máquina reparte premios según la misma distribución fija: no cambia con el tiempo ni reacciona a tus decisiones anteriores
- Por eso es un problema "sin estado": lo único que cambia mientras juegas es tu propio conocimiento, no el entorno

### La rata y las dos palancas

Piensa en una rata de laboratorio que puede escoger entre dos palancas, una sobre fondo negro y otra sobre fondo blanco. Un día, la palanca negra da queso; otro día, da una descarga. Si la rata se guiara solo por el último resultado, o por una sola mala experiencia, podría abandonar para siempre una palanca que, en promedio, es la mejor opción.

Lo mismo le pasa a cualquier agente que decide de forma demasiado precipitada: **una mala racha inicial puede convencerlo de evitar la mejor acción posible durante el resto del juego**.


## Tres. Estimando el valor de cada acción

> Guion: pregunta al grupo cómo calcularían el valor de una palanca si solo pudieran anotar resultados en una libreta. Suele salir la media.

Si no conoces el valor real de una palanca, lo razonable es estimarlo a partir de lo que has observado. La forma más directa es calcular la media de las recompensas que has recibido cada vez que elegiste esa acción. A esa estimación se le llama **valor de acción**, y se actualiza cada vez que vuelves a probar la misma palanca.

No hace falta guardar todo el historial de recompensas para calcular esa media. Puedes actualizarla de forma **incremental**: cada vez que observas una recompensa nueva, ajustas tu estimación anterior un poco hacia ese nuevo valor, con un peso que disminuye cuanto más veces hayas probado esa acción:

$$Q_{n+1} = Q_n + \frac{1}{n}(R_n - Q_n)$$

Al principio, cada observación nueva cambia mucho tu estimación; con el tiempo, cada observación adicional pesa menos y tu estimación se estabiliza cerca del valor real.

**Variante útil:** en vez de dar el mismo peso a todas las observaciones pasadas, puedes usar una tasa de paso fija (un "learning rate" constante) que dé más peso a las recompensas recientes. Es especialmente útil en entornos no estacionarios, donde una máquina que hoy paga bien puede pagar peor mañana.


## Cuatro. Epsilon-greedy: explorar un poco, explotar mucho

> Guion: pide a alguien del grupo que proponga un valor de ε y calcula en voz alta qué porcentaje de veces exploraría.

La estrategia más sencilla para resolver el dilema se llama **epsilon-greedy** (ε-greedy). Funciona así:

- Define un número pequeño, épsilon (ε), entre cero y uno
- En cada decisión, con probabilidad ε eliges una acción al azar (**exploras**)
- Con probabilidad 1−ε, eliges la acción que hasta ahora parece mejor según tus estimaciones (**explotas**)

Un detalle curioso: como la elección aleatoria también puede coincidir con la mejor acción conocida, la probabilidad real de explotar es ligeramente mayor que 1−ε.

Volvamos al ejemplo del restaurante. Si tu ε es 0,1, entonces el 90% de las veces vas a tu sitio de confianza y el 10% de las veces pruebas algo distinto, elegido al azar entre todas las opciones disponibles.

Un ε fijo garantiza que sigas explorando para siempre: con el tiempo acabarás probando todas las acciones las veces suficientes para conocer su valor real con bastante precisión. Pero tiene un coste: **seguirás cometiendo el mismo porcentaje de errores** incluso después de haber aprendido cuál es la mejor opción. Por eso, una mejora habitual es hacer que ε decrezca con el tiempo. Esta estrategia se usó, por ejemplo, en Deep Q-Network (DQN), uno de los sistemas que aprendió a jugar videojuegos de Atari directamente a partir de los píxeles de la pantalla.


## Cinco. Upper Confidence Bound: el optimismo ante la incertidumbre

> Guion: introduce la frase "optimismo ante la incertidumbre" y pide que la relacionen con la vida real (probar un proveedor nuevo, una receta nueva).

Epsilon-greedy tiene una limitación: cuando explora, elige al azar entre todas las acciones, sin distinguir entre una palanca que ya conoces bien y otra que apenas has probado. **Upper Confidence Bound (UCB)** resuelve esto de una forma más inteligente.

La lógica de UCB: en vez de mirar solo tu estimación actual del valor de cada acción, súmale un **bono de incertidumbre**:

- Es grande para las acciones que has probado pocas veces (les das el beneficio de la duda)
- Es pequeño para las que has probado muchas veces (ya tienes una estimación fiable)
- En cada paso, eliges la acción cuya suma de **valor estimado + bono** sea la más alta

Al principio, como no has probado ninguna palanca, el bono de las acciones sin probar es enorme, así que UCB te obliga a probar cada acción al menos una vez antes de empezar a comparar. A medida que juegas, el bono de cada palanca se reduce conforme la pruebas más, siguiendo aproximadamente el ritmo de la raíz cuadrada del número de veces que la has intentado. Así, el algoritmo pasa de forma natural de una fase de mucha exploración a una fase centrada en explotar lo que ya sabe.

Existen variantes de UCB para distintos escenarios: UCB1-Tuned ajusta el bono según cuánto varían las recompensas observadas; UCB Descontado da más peso a las observaciones recientes; UCB Contextual añade información sobre la situación (por ejemplo, características del usuario en un sistema de recomendación).


## Seis. Thompson Sampling: una alternativa bayesiana

> Guion: si vas justa o justo de tiempo, esta sección puede resumirse en dos minutos; el foco de la práctica de hoy es epsilon-greedy y UCB.

Existe una tercera familia de estrategias, de corte bayesiano: **Thompson Sampling** (también llamado muestreo de Thompson o bandido bayesiano). En vez de guardar una sola estimación por acción, mantiene una **distribución de probabilidad completa** sobre el valor de cada acción: una especie de "creencia" que refleja tanto tu mejor estimación como tu incertidumbre sobre ella.

En cada paso:

1. Para cada acción, obtén un valor de muestra a partir de su distribución de creencia actual
2. Elige la acción con la muestra más alta
3. Observa la recompensa real y actualiza la creencia de esa acción

Cuando una acción tiene una distribución muy incierta, sus muestras varían mucho y a veces salen valores altos: eso es exploración. Cuando una distribución ya está consolidada en torno a un valor alto, sus muestras son casi siempre altas y consistentes: eso es explotación. Todo ocurre de forma natural, sin necesitar un parámetro como ε ni una fórmula de bono explícita.

Para recompensas binarias (éxito o fracaso), Thompson Sampling suele usar la distribución Beta: empiezas con una distribución neutra y, cada vez que observas un éxito, ajustas un parámetro; cada vez que observas un fracaso, ajustas el otro.


## Siete. Código en vivo: epsilon-greedy contra UCB

> Guion: a partir de aquí, ejecuta cada celda por separado y comenta el resultado antes de pasar a la siguiente. Es el mismo código del manual, dividido en pasos.

Vas a simular un bandido de varios brazos y a comparar, de forma visual, cómo se comporta epsilon-greedy frente a UCB a lo largo de muchos pasos. Solo necesitas `numpy` y `matplotlib`, que ya vienen preinstalados en Google Colab.

**Ejecuta cada celda en orden.**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Fijamos una semilla para que el experimento sea reproducible
np.random.seed(42)


### Paso uno: configura el bandido

Define cuántos brazos tiene el bandido, cuántos pasos va a jugar cada estrategia y cuántas veces repites el experimento completo para suavizar el efecto del azar.


In [ ]:
# --- Configuración del bandido multi-brazo ---
n_brazos = 10          # número de "palancas" o acciones disponibles
n_pasos = 1000          # cuántas veces va a jugar cada estrategia
n_repeticiones = 200    # repetimos el experimento varias veces para suavizar el azar

# El valor verdadero de cada brazo se genera al azar (el agente NO lo conoce)
# Estos valores representan la recompensa media real de cada acción
valores_reales = np.random.normal(loc=0.0, scale=1.0, size=n_brazos)
valores_reales


### Paso dos: simula tirar de una palanca

Cada vez que el agente elige un brazo, recibe una recompensa con ruido alrededor del valor real de ese brazo, tal y como pasaría en una tragaperras real.


In [ ]:
def tirar_de_la_palanca(brazo):
    """Simula el resultado de elegir un brazo: devuelve una recompensa con ruido
    alrededor del valor verdadero de ese brazo (como en una tragaperras real)."""
    return np.random.normal(loc=valores_reales[brazo], scale=1.0)


# Prueba rápida: tira varias veces del brazo 0 y observa la variación
[round(tirar_de_la_palanca(0), 2) for _ in range(5)]


### Paso tres: implementa epsilon-greedy

Con probabilidad ε explora al azar; el resto de las veces explota la mejor acción conocida hasta ahora.


In [ ]:
def ejecutar_epsilon_greedy(epsilon, n_pasos):
    """Estrategia epsilon-greedy: explora al azar con probabilidad epsilon,
    y el resto de las veces elige la acción con mejor valor estimado."""
    valores_estimados = np.zeros(n_brazos)   # Q_t(a): estimación actual de cada brazo
    conteo_intentos = np.zeros(n_brazos)     # cuántas veces se ha probado cada brazo
    recompensas = np.zeros(n_pasos)

    for t in range(n_pasos):
        if np.random.random() < epsilon:
            # Exploración: elige un brazo al azar
            brazo = np.random.randint(n_brazos)
        else:
            # Explotación: elige el brazo con mejor valor estimado hasta ahora
            brazo = np.argmax(valores_estimados)

        recompensa = tirar_de_la_palanca(brazo)
        conteo_intentos[brazo] += 1

        # Actualización incremental de la media muestral (sección tres):
        # Q_{n+1} = Q_n + (1/n) * (R_n - Q_n)
        paso = 1.0 / conteo_intentos[brazo]
        valores_estimados[brazo] += paso * (recompensa - valores_estimados[brazo])

        recompensas[t] = recompensa

    return recompensas


### Paso cuatro: implementa UCB

Combina el valor estimado con un bono de incertidumbre que favorece a los brazos poco probados.


In [ ]:
def ejecutar_ucb(c, n_pasos):
    """Estrategia UCB: combina el valor estimado con un bono de incertidumbre
    que favorece a los brazos poco probados."""
    valores_estimados = np.zeros(n_brazos)
    conteo_intentos = np.zeros(n_brazos)
    recompensas = np.zeros(n_pasos)

    for t in range(1, n_pasos + 1):
        # Si algún brazo aún no se ha probado, se elige primero (bono "infinito")
        brazos_sin_probar = np.where(conteo_intentos == 0)[0]
        if len(brazos_sin_probar) > 0:
            brazo = brazos_sin_probar[0]
        else:
            # UCB: valor estimado + bono de exploración que crece con log(t)
            # y se reduce con la raíz cuadrada del número de intentos del brazo
            bono = c * np.sqrt(np.log(t) / conteo_intentos)
            brazo = np.argmax(valores_estimados + bono)

        recompensa = tirar_de_la_palanca(brazo)
        conteo_intentos[brazo] += 1
        paso = 1.0 / conteo_intentos[brazo]
        valores_estimados[brazo] += paso * (recompensa - valores_estimados[brazo])

        recompensas[t - 1] = recompensa

    return recompensas


### Paso cinco: ejecuta ambas estrategias muchas veces

Repites el experimento 200 veces, regenerando los valores reales en cada repetición, para que el promedio no dependa de una única configuración de brazos.

> Guion: esta celda tarda unos segundos. Aprovecha para explicar por qué repetimos el experimento en vez de correrlo una sola vez.


In [ ]:
# --- Ejecutamos ambas estrategias varias veces y promediamos ---
recompensas_epsilon = np.zeros(n_pasos)
recompensas_ucb = np.zeros(n_pasos)

for _ in range(n_repeticiones):
    # Re-generamos los valores reales en cada repetición para que el promedio
    # no dependa de una única configuración de brazos
    valores_reales = np.random.normal(loc=0.0, scale=1.0, size=n_brazos)
    recompensas_epsilon += ejecutar_epsilon_greedy(epsilon=0.1, n_pasos=n_pasos)
    recompensas_ucb += ejecutar_ucb(c=2.0, n_pasos=n_pasos)

recompensas_epsilon /= n_repeticiones
recompensas_ucb /= n_repeticiones

print("Listo: experimento completado")


### Paso seis: calcula la recompensa media acumulada

Convierte la recompensa obtenida en cada paso en una recompensa media acumulada, para ver la tendencia de aprendizaje de cada estrategia.


In [ ]:
# Convertimos la recompensa por paso en recompensa media acumulada
recompensa_media_acumulada_epsilon = np.cumsum(recompensas_epsilon) / np.arange(1, n_pasos + 1)
recompensa_media_acumulada_ucb = np.cumsum(recompensas_ucb) / np.arange(1, n_pasos + 1)

print("Recompensa media acumulada final, epsilon-greedy:", round(recompensa_media_acumulada_epsilon[-1], 3))
print("Recompensa media acumulada final, UCB:", round(recompensa_media_acumulada_ucb[-1], 3))


### Paso siete: grafica la comparación

> Guion: antes de ejecutar, pregunta al grupo qué curva creen que va a ir por delante y por qué.


In [ ]:
# --- Graficamos la comparación ---
plt.figure(figsize=(10, 5))
plt.plot(recompensa_media_acumulada_epsilon, label="epsilon-greedy (ε = 0.1)")
plt.plot(recompensa_media_acumulada_ucb, label="UCB (c = 2.0)")
plt.xlabel("Paso de tiempo")
plt.ylabel("Recompensa media acumulada")
plt.title("Comparación epsilon-greedy vs. UCB en un bandido de 10 brazos")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


### Qué observar en la gráfica

Ambas curvas mejoran con el tiempo, pero **UCB suele adelantarse a epsilon-greedy**: como reparte su exploración de forma más inteligente (probando primero lo que menos conoce), tarda menos en encontrar el brazo óptimo y desperdicia menos intentos en brazos claramente malos.


## Puntos clave de la clase

- El dilema entre exploración y explotación aparece en cualquier decisión bajo incertidumbre, dentro y fuera del RL
- El bandido multi-brazo es la versión más simple de este problema: varias acciones, cada una con un valor real desconocido y fijo en el tiempo
- El valor de una acción se estima con la media de las recompensas observadas, y se puede actualizar de forma incremental sin guardar todo el historial
- Epsilon-greedy explora al azar con probabilidad ε y explota el resto del tiempo; un ε decreciente suele funcionar mejor que uno fijo
- UCB dirige la exploración hacia las acciones más inciertas, sumando un bono que se reduce a medida que una acción se prueba más
- Thompson Sampling usa distribuciones de probabilidad completas y equilibra exploración y explotación de forma natural, sin parámetros explícitos como ε

## Siguiente paso

**¡Abre ahora el notebook de práctica** (`clase02_practica_bandidos.ipynb`) **y completa los dos ejercicios guiados!** Vas a implementar UCB desde cero y a comparar distintos valores de ε.

En la clase 3 vas a dejar atrás el mundo "sin estado" de los bandidos para entrar en los Procesos de Decisión de Markov, donde las acciones del agente sí cambian el estado del entorno.
